# J-Space top-k tokens by layer

Reads saved capture files (no model load). For one example and token position, shows:

- the top-k J-Space vocabulary at each captured layer
- which layers each of those tokens appears in

Recording is chosen at **run time**, not in this notebook:

```bash
python -m gsm8k_jspace run --config configs/runs/small-smoke.yaml --capture
python -m gsm8k_jspace run --config configs/smoke.yaml --no-capture
```

YAML equivalent: `capture.enabled: true` or `false`, or overlays
`configs/experiments/capture-on.yaml` / `capture-off.yaml`.

Set `SAVE_OUTPUTS = True` below to write tables/figures into the run folder.
Use `SOURCE = "logit"` or `"model"` when the run stored those readouts (`full_sequence` capture).

In [6]:
RUN_DIR = "m1max-qwen35-4b-all-layers-infer_20260820T010639Z"
EXAMPLE_ID = None
POSITION = "last"
SOURCE = "jspace"
MAX_RANK = 10
SAVE_OUTPUTS = False

In [7]:
from IPython.display import Markdown, display
from pathlib import Path
import pandas as pd

from gsm8k_jspace.analysis import (
    load_run,
    plot_token_layer_heatmap,
    token_layer_presence_table,
    token_layer_rank_grid,
    topk_by_layer_table,
)
from gsm8k_jspace.analysis.catalog import resolve_run

run_path = resolve_run(RUN_DIR)
run = load_run(run_path)
warnings = run.warn_incomplete()
example_id = EXAMPLE_ID or (
    run.completions[0]["example_id"] if run.completions else None
)
completion = next(
    (row for row in run.completions if row.get("example_id") == example_id),
    {},
)
recorded = run.has_captures()
save_outputs = bool(SAVE_OUTPUTS)

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

example_widget = None
source_widget = None
position_widget = None
save_widget = None
if widgets is not None:
    example_options = [row["example_id"] for row in run.completions] or [example_id]
    example_widget = widgets.Dropdown(
        options=example_options,
        value=example_id if example_id in example_options else example_options[0],
        description="Example",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "90px"},
    )
    source_widget = widgets.Dropdown(
        options=["jspace", "logit", "model"],
        value=SOURCE,
        description="Source",
        style={"description_width": "90px"},
    )
    position_widget = widgets.Dropdown(
        options=["last", "all"],
        value=POSITION if POSITION in {"last", "all"} else "last",
        description="Position",
        style={"description_width": "90px"},
    )
    save_widget = widgets.Checkbox(
        value=save_outputs,
        description="Save tables/figures into this run folder",
        indent=False,
    )
    display(example_widget, source_widget, position_widget, save_widget)
    display(Markdown("_Change a control, then re-run the cells below._"))

meta = pd.DataFrame(
    [
        {
            "Run": run.manifest.get("run_id"),
            "Status": run.manifest.get("status"),
            "Example": example_id,
            "Position": POSITION,
            "Source": SOURCE,
            "Top-k": MAX_RANK,
            "Capture recorded": recorded,
            "capture.enabled": run.config.capture.enabled,
            "Capture file": completion.get("capture_file"),
            "Save outputs": save_outputs,
            "Warnings": "; ".join(warnings) if warnings else "none",
        }
    ]
)
display(Markdown(f"**Run** `{run_path}`"))
display(meta.style.hide(axis="index"))
if not recorded:
    display(
        Markdown(
            "**This run did not record J-Space captures.** "
            "GSM8K completions may still be here. To record top-k tokens by layer, re-run with "
            "`--capture` or `capture.enabled: true`."
        )
    )
if completion.get("generated_text"):
    display(Markdown("**Generated text**"))
    display(Markdown(f"```\n{completion['generated_text']}\n```"))


def _selected():
    chosen_example = example_widget.value if example_widget is not None else example_id
    chosen_source = source_widget.value if source_widget is not None else SOURCE
    chosen_position = position_widget.value if position_widget is not None else POSITION
    chosen_save = save_widget.value if save_widget is not None else save_outputs
    return chosen_example, chosen_source, chosen_position, chosen_save


def _output_dirs():
    tables = run.run_dir / "visualization" / "tables"
    figures = run.run_dir / "visualization" / "figures"
    tables.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return tables, figures

Dropdown(description='Example', layout=Layout(width='100%'), options=('gsm8k_test_000000', 'gsm8k_test_000001'…

Dropdown(description='Source', options=('jspace', 'logit', 'model'), style=DescriptionStyle(description_width=…

Dropdown(description='Position', options=('last', 'all'), style=DescriptionStyle(description_width='90px'), va…

Checkbox(value=False, description='Save tables/figures into this run folder', indent=False)

_Change a control, then re-run the cells below._

**Run** `/Users/cyb/Documents/GitHub/jspace-reserach-qwen/qwen-gsm8k-jlens/outputs/gsm8k/m1max-qwen35-4b-all-layers-infer_20260820T010639Z`

Run,Status,Example,Position,Source,Top-k,Capture recorded,capture.enabled,Capture file,Save outputs,Warnings
m1max-qwen35-4b-all-layers-infer_20260820T010639Z,complete,gsm8k_test_000000,last,jspace,10,True,True,captures/gsm8k_test_000000.jsonl.gz,False,none


**Generated text**

```
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Solve a grade-school math problem.
    *   Requirement: Show reasoning, then put the final numeric answer after `####`.
    *   Problem Statement: Janet's ducks lay 16 eggs per day. She eats 3 for breakfast. She bakes muffins with 4. She sells the remainder at $2 per egg. How much does she make at the farmers' market daily?

2.  **Break Down the Problem:**
    *   Total eggs produced per day = 16.
    *   Eggs used for breakfast = 3.
    *   Eggs used for muffins = 4.
    *   Selling price per egg = $2.
    *   Goal: Calculate total earnings from selling the remaining eggs.

3.  **Step-by-Step Calculation:**
    *   Step 1: Calculate the number of eggs remaining after breakfast and muffins.
        *   Remaining eggs = Total eggs - (Eggs for breakfast + Eggs for muffins)
        *   Remaining eggs = 16 - (3 + 4)
        *   Remaining eggs = 1
```

In [ ]:
example_id, source, position, save_outputs = _selected()
by_layer = pd.DataFrame(
    topk_by_layer_table(
        run,
        example_id=example_id,
        position=position,
        source=source,
        max_rank=MAX_RANK,
    )
)
if by_layer.empty:
    display(
        Markdown(
            "No top-k rows. This run may have used `--no-capture`, or it did not store "
            f"`top_{source}_tokens` at the selected position."
        )
    )
else:
    state = by_layer["state_token"].iloc[0]
    pos = by_layer["position"].iloc[0]
    display(
        Markdown(
            f"**Top-{MAX_RANK} {source} tokens at position `{pos}`** "
            f"(state token {state}, one row per layer)"
        )
    )
    display(by_layer.drop(columns=["position", "state_token"]).style.hide(axis="index"))
    if save_outputs:
        tables_dir, _ = _output_dirs()
        path = tables_dir / "topk_by_layer.csv"
        by_layer.to_csv(path, index=False)
        display(Markdown(f"Saved `{path}`"))

**Top-10 jspace tokens at position `358`** (state token , one row per layer)

layer,rank_1,rank_2,rank_3,rank_4,rank_5,rank_6,rank_7,rank_8,rank_9,rank_10
0,""" --"" (9.45)",""" )."" (9.16)",""" ,"" (8.73)",""" ``"" (8.41)",""" 、"" (8.24)",""" -"" (7.82)","""ly"" (7.63)","""s"" (7.53)","""istic"" (7.51)",""" in"" (7.47)"
1,""" ``"" (11.66)",""" --"" (9.02)","""ly"" (8.80)",""" ,"" (8.20)","""ers"" (8.18)","""ed"" (7.77)","""s"" (7.66)",""" â"" (7.50)",""" :"" (7.37)",""" <"" (7.34)"
2,""","" (14.21)","""."" (13.17)","""s"" (12.72)","""ly"" (11.27)",""" at"" (10.98)",""" in"" (10.91)","""a"" (10.64)","""\n"" (10.61)","""ed"" (10.54)",""" the"" (10.27)"
3,"""ly"" (13.01)","""ers"" (12.33)","""s"" (11.43)",""""" (11.21)","""er"" (11.02)","""ed"" (10.94)",""" --"" (10.76)","""."" (10.61)","""a"" (10.07)","""ized"" (10.03)"
4,"""."" (19.67)",""","" (15.44)","""ly"" (14.96)","""ers"" (14.00)","""s"" (13.84)","""ed"" (13.25)","""er"" (13.21)","""a"" (12.82)",""".."" (12.79)","""o"" (12.68)"
5,"""ly"" (15.98)","""."" (15.60)","""ers"" (14.97)","""..."" (14.40)",""".."" (13.97)","""er"" (13.62)","""s"" (13.55)","""o"" (13.50)","""a"" (13.14)","""ed"" (12.85)"
6,"""ly"" (17.47)","""ers"" (15.87)","""ed"" (14.01)","""er"" (13.90)","""o"" (13.34)","""s"" (13.05)","""a"" (12.82)","""en"" (12.62)","""ized"" (12.48)","""istic"" (12.44)"
7,"""ly"" (17.72)","""ers"" (16.42)","""s"" (14.76)","""er"" (14.53)","""ed"" (14.50)","""o"" (13.93)","""."" (13.64)","""a"" (13.47)","""n"" (13.26)","""ized"" (12.77)"
8,"""ly"" (16.48)","""ers"" (16.11)","""\n"" (14.03)","""."" (13.80)","""ed"" (13.71)","""er"" (13.63)","""\\"" (13.61)","""..."" (13.54)","""s"" (13.44)",""" \\"" (13.24)"
9,"""..."" (17.58)","""\n"" (15.94)","""\n\n"" (15.90)","""ly"" (15.18)",""" \\"" (15.17)",""""" (14.85)",""" ..."" (14.57)","""ers"" (14.35)","""\\n"" (14.18)","""\\"" (14.12)"


In [ ]:
example_id, source, position, save_outputs = _selected()
presence = pd.DataFrame(
    token_layer_presence_table(
        run,
        example_id=example_id,
        position=position,
        source=source,
        max_rank=MAX_RANK,
    )
)
if presence.empty:
    display(Markdown("No token-to-layer mapping for this selection."))
else:
    shown = presence.rename(
        columns={
            "token": "Token",
            "token_id": "Token id",
            "n_layers": "Layers",
            "layers": "Appears in layers",
            "best_rank": "Best rank",
            "mean_logit": "Mean logit",
            "n_hits": "Hits",
        }
    )
    if "Mean logit" in shown:
        shown["Mean logit"] = pd.to_numeric(shown["Mean logit"]).map(
            lambda value: f"{value:.2f}" if pd.notna(value) else ""
        )
    display(
        Markdown(
            "**Which layers each top-k token appears in** "
            "(sorted by best rank, then how many layers)"
        )
    )
    display(shown.style.hide(axis="index"))
    if save_outputs:
        tables_dir, _ = _output_dirs()
        path = tables_dir / "topk_token_layers.csv"
        presence.to_csv(path, index=False)
        display(Markdown(f"Saved `{path}`"))

**Which layers each top-k token appears in** (sorted by best rank, then how many layers)

Token,Token id,Layers,Appears in layers,Best rank,Mean logit,Hits
"""ly""",391,14,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15",1,13.70,14
""" minus""",26401,10,"19, 20, 21, 22, 23, 24, 25, 26, 27, 28",1,15.98,10
""".""",13,10,"2, 3, 4, 5, 7, 8, 13, 15, 16, 17",1,14.96,10
"""...""",1076,8,"5, 8, 9, 10, 11, 12, 13, 14",1,15.01,8
"""6""",21,7,"17, 18, 26, 27, 28, 29, 30",1,18.06,7
"""\\n""",1639,5,"9, 10, 13, 14, 15",1,14.17,5
"""3""",18,4,"17, 18, 29, 30",1,14.69,4
""" --""",1137,4,"0, 1, 3, 11",1,10.50,4
""",""",11,2,"2, 4",1,14.82,2
""" ``""",9609,2,"0, 1",1,10.03,2


In [ ]:
example_id, source, position, save_outputs = _selected()
grid = token_layer_rank_grid(
    run,
    example_id=example_id,
    position=position,
    source=source,
    max_rank=MAX_RANK,
)
grid_df = pd.DataFrame(grid)
if grid_df.empty:
    display(Markdown("No rank grid to plot."))
else:
    rank_cols = [col for col in grid_df.columns if col.startswith("L") and col[1:].isdigit()]
    display(Markdown("**Rank grid** (blank = token not in that layer's top-k)"))
    display(
        grid_df[["token", "best_rank", "n_layers", *rank_cols]]
        .rename(columns={"token": "Token", "best_rank": "Best rank", "n_layers": "Layers"})
        .style.hide(axis="index")
        .background_gradient(cmap="Blues_r", subset=rank_cols, vmin=1, vmax=MAX_RANK)
    )
    fig_path = None
    if save_outputs:
        _, figures_dir = _output_dirs()
        fig_path = figures_dir / "topk_rank_by_layer.png"
        tables_dir, _ = _output_dirs()
        grid_df.to_csv(tables_dir / "topk_rank_grid.csv", index=False)
    fig = plot_token_layer_heatmap(
        grid,
        title=f"{source} top-{MAX_RANK} rank by layer ({example_id})",
        max_tokens=40,
        path=fig_path,
    )
    display(fig)
    if fig_path is not None:
        display(Markdown(f"Saved `{fig_path}`"))

**Rank grid** (blank = token not in that layer's top-k)

ImportError: `Import matplotlib` failed. Styler.background_gradient requires matplotlib. Use pip or conda to install the matplotlib package.

RuntimeError: matplotlib is required for plotting